# BreastMNIST analysis using PYTORCH

### **Step 1: Package Import**

In [41]:
from importlib.metadata import version
import os

packages = ["torch", "medmnist", "numpy", "pandas", "scikit-learn", "matplotlib", "ipykernel"]

for package in packages:
    print(package + "==" + version(package))

torch==2.15.0
medmnist==3.0.2
numpy==2.5.4
pandas==3.1.1
scikit-learn==1.9.1
matplotlib==3.11.2
ipykernel==7.4.0


### **Step 2: Data Download**

Based on the official MedMNIST documentation:

- The file ./data/breastmnist.npz holds six arrays: one for images 
and one for labels for each of the training, validation and test splits.

- Images are grayscale and 28 by 28 pixels, so the training images should 
have the shape (546, 28, 28) and the training labels the shape (546, 1).

- Each label is a single number, 0 or 1, because this is a two-class task.

- The standard split is already provided: 546 training, 78 validation and
156 test images (780 in total),  by the MedMNIST authors, so that everyone 
uses the same test images and results can be compared. The data is for 
learning and research only, not for clinical use.

In [42]:
import os
from medmnist import BreastMNIST

os.makedirs("data", exist_ok=True)

train_data = BreastMNIST(split="train", download=True, root="data")
validation_data = BreastMNIST(split="val", download=True, root="data")
test_data = BreastMNIST(split="test", download=True, root="data")

# to verify the split numbers
print("Training images:  ", len(train_data))
print("Validation images:", len(validation_data))
print("Test images:      ", len(test_data))

Training images:   546
Validation images: 78
Test images:       156


Looking inside the data file itself, it is a bundle of the **six mentioned arrays**, called: 

- train_images, val_images, test_images: the pictures. 

- train_labels, val_labels, test_labels: the answers. Each has the shape
(number of images, 1), meaning one label per picture, in the same order as
the images. 

Expected printout: **three image arrays** with 546, 78 and 156 pictures,
and **three label arrays** with the *same counts and a second dimension of 1*.

In [43]:
file_contents = numpy.load("data/breastmnist.npz")

for name in file_contents.files:
    print(name, file_contents[name].shape)

train_images (546, 28, 28)
val_images (78, 28, 28)
test_images (156, 28, 28)
train_labels (546, 1)
val_labels (78, 1)
test_labels (156, 1)


Below, I want to check the **split ratios** among the train, validation and test split groups:

In [44]:
sizes = {
    "training": len(train_data),
    "validation": len(validation_data),
    "test": len(test_data),
}
total = sum(sizes.values())

print("Total images:", total)
for name, size in sizes.items():
    print(name, size, "images,", round(100 * size / total, 1), "percent")

Total images: 780
training 546 images, 70.0 percent
validation 78 images, 10.0 percent
test 156 images, 20.0 percent


In [53]:
# The names of the 0 vs 1 labels come from the medmnist package itself, and have specific names.
label_names = medmnist.INFO["breastmnist"]["label"]

# For each split: taking its labels, flattening them into one list (.ravel()),
# and counting how many images have each label (.value_counts()).
label_table = pandas.DataFrame({
    "training": pandas.Series(train_data.labels.ravel()).value_counts(),
    "validation": pandas.Series(validation_data.labels.ravel()).value_counts(),
    "test": pandas.Series(test_data.labels.ravel()).value_counts(),
}).sort_index()

# Adding a total column, and writing each label as "number = name".
label_table["total"] = label_table.sum(axis=1)
label_table.index = [str(label) + " = " + label_names[str(label)] for label in label_table.index]

print("Number of images:")
print(label_table)
print()

Number of images:
                    training  validation  test  total
0 = malignant            147          21    42    210
1 = normal, benign       399          57   114    570



### **Step 3: Metadata Download**

`breastmnist_split_info.csv` is a table provided by the MedMNIST authors. For each of the 780 images in BreastMNIST it gives the split (training, validation or test), the position of the image inside that split, and the name of the image in the original dataset.

The processed BreastMNIST arrays carry only a two-class label. This file includes the original tumor category (normal, benign or malignant) and the original image name of each picture. It is a nice-to-have for potenntial downstream review of the results.

In [47]:
# Loading the metadata file and looking at its structure.
metadata = pandas.read_csv("metadata/breastmnist_split_info.csv")


print("Rows and columns:", metadata.shape)
print()

# Checking the first and last 5 rows, to have an initial overview.
print("First 5 rows:")
print(metadata.head(5).to_string())
print()
print("Last 5 rows:")
print(metadata.tail(5).to_string())

Rows and columns: (780, 3)

First 5 rows:
   split  index         image_id
0  train      0      normal (45)
1  train      1      benign (56)
2  train      2     benign (244)
3  train      3     benign (325)
4  train      4  malignant (163)

Last 5 rows:
    split  index         image_id
775  test    151      normal (11)
776  test    152     benign (268)
777  test    153  malignant (209)
778  test    154      benign (77)
779  test    155     benign (308)


In [48]:
# Checking empty cells per column. Zero everywhere means every row is complete.
print("Empty cells per column:")
print(metadata.isna().sum())
print()

# Checking how many images each split has (the counts the notebook printed in Step 2 should agree).
print("Images per split:")
print(metadata["split"].value_counts())
print()

# Checking for duplicate image names.
# .duplicated() marks every image name that already appeared earlier in the table.
# Zero means every row has its own distinct original image name.
print("Image names that appear more than once:", int(metadata["image_id"].duplicated().sum()))

Empty cells per column:
split       0
index       0
image_id    0
dtype: int64

Images per split:
split
train    546
test     156
val       78
Name: count, dtype: int64

Image names that appear more than once: 0


In [49]:
# Splitting each image name into its tumor category and number, then count.
# This adds columns to the table in memory only; the file on disk is not changed.
# .str.extract reads each name such as "benign (56)" with a pattern: the word before
# the space becomes column 0 and the number inside the brackets becomes column 1.
extracted = metadata["image_id"].str.extract(r"^(\w+) \((\d+)\)$")
metadata["original_class"] = extracted[0]                 # normal, benign or malignant
metadata["original_number"] = extracted[1].astype(int)    # the number as a whole number

# Names that did not fit the pattern would be empty in column 0; zero is expected.
print("Rows where the name did not fit the pattern:", int(extracted[0].isna().sum()))
print()

# Making a table with one row per category and one column per split,
# each cell counting the images with that combination.
# margins=True adds totals; the last list puts the columns in a logical order.
print(pandas.crosstab(metadata["original_class"], metadata["split"], margins=True, margins_name="total")[["train", "val", "test", "total"]])

Rows where the name did not fit the pattern: 0

split           train  val  test  total
original_class                         
benign            307   39    91    437
malignant         146   21    43    210
normal             93   18    22    133
total             546   78   156    780


In [50]:
# Checking that the metadata file lines up with the BreastMNIST arrays.
# What it does: for every row of the metadata table, it looks up the two-class label (0 or 1)
# of the image at that split and position in the arrays, and adds it as a new column.
# Then it counts how the original categories (normal, benign, malignant) fall under 0 and 1.
# Goal: to verify that the metadata and the arrays are in sync, and that the original categories


# The label arrays of the three splits, gathered under the same names the metadata uses.
# Each array has the shape (number of images, 1), so one image's label is labels[position][0].
labels_by_split = {
    "train": train_data.labels,
    "val": validation_data.labels,
    "test": test_data.labels,
}

# For each metadata row: take its split and its position, and read the label at that place.
# zip walks through the two columns at the same time, row by row.
# int(...) turns the single array value into a plain whole number.
metadata["binary_label"] = [
    int(labels_by_split[split_name][position][0])
    for split_name, position in zip(metadata["split"], metadata["index"])
]

# Count the images for every combination of split, original category and label.
print(pandas.crosstab([metadata["split"], metadata["original_class"]], metadata["binary_label"]))

binary_label            0    1
split original_class          
test  benign            0   91
      malignant        42    1
      normal            0   22
train benign            1  306
      malignant       146    0
      normal            0   93
val   benign            0   39
      malignant        21    0
      normal            0   18


In [51]:
# Malignant images should all be under 0, and benign and normal images all under 1.
# Nearly everything follows that pattern, but two images don’t.

# List the rows that break the expected pattern:
# malignant should have label 0, normal and benign should have label 1.
mismatch = metadata[
    ((metadata["original_class"] == "malignant") & (metadata["binary_label"] == 1))
    | ((metadata["original_class"] != "malignant") & (metadata["binary_label"] == 0))
]
print(mismatch.to_string())

     split  index         image_id original_class  original_number  binary_label
115  train    115     benign (433)         benign              433             0
700   test     76  malignant (145)      malignant              145             1


In [52]:
# Checking if the two flagged images are the same picture
# array_equal is True only if every pixel is identical.
print(numpy.array_equal(train_data.imgs[115], test_data.imgs[76]))

True


This one picture sits in both the training and test sets with opposite labels, which shall be noted for the downstream analysis.